# Aprendizagem de Máquina - Web Academy

## Regressão e Comparação de Classificadores

---

## Parte 1: Dominando a Regressão – Prevendo Preços de Aluguel de Imóveis

Nesta primeira parte, o foco será um problema clássico e de grande valor comercial: a previsão de preços. O objetivo é construir um modelo de regressão robusto, passo a passo.

### 1.1. O Problema de Negócio e Nossos Dados

- **Contexto:** Imagine uma startup de tecnologia imobiliária (*proptech*) que deseja adicionar uma nova funcionalidade à sua plataforma: um estimador instantâneo de aluguel para imóveis no Brasil. 
- **Seleção do Dataset:** Para este projeto, será utilizado o dataset "Brazilian Houses to Rent", disponível no Kaggle. 
- **Exploração Inicial dos Dados:** O primeiro passo é carregar o dataset usando a biblioteca `pandas` e inspecionar sua estrutura.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Carregar o dataset
# O arquivo CSV pode ser baixado do Kaggle: https://www.kaggle.com/datasets/rubenssjr/brasilian-houses-to-rent
# Certifique-se de que o arquivo 'houses_to_rent_v2.csv' está na mesma pasta que este notebook.
df_aluguel = pd.read_csv('houses_to_rent_v2.csv')

# Exibir as primeiras 5 linhas
print("Amostra do Dataset de Aluguel:")
display(df_aluguel.head())

# Exibir informações sobre as colunas e tipos de dados
print("\nInformações Gerais do Dataset:")
df_aluguel.info()

#### Análise das Colunas

As colunas presentes no dataset são:

* `city`: A cidade onde o imóvel está localizado.
* `area`: A área do imóvel em metros quadrados.
* `rooms`: Número de quartos.
* `bathroom`: Número de banheiros.
* `parking spaces`: Número de vagas de garagem.
* `floor`: O andar em que o apartamento está localizado.
* `animal`: Se animais de estimação são permitidos ('acept' ou 'not acept').
* `furniture`: Se o imóvel é mobiliado ('furnished' ou 'not furnished').
* `hoa (R$)`: Taxa de condomínio (mensal).
* `rent amount (R$)`: O valor do aluguel mensal (nossa variável alvo).
* `property tax (R$)`: Imposto predial (mensal).
* `fire insurance (R$)`: Seguro contra incêndio (mensal).
* `total (R$)`: O custo total mensal.

#### A Importância Crítica de Definir a Variável Alvo e Identificar Vazamento de Dados (Data Leakage)

- **O Problema:** A coluna `total (R$)` é a soma de `rent amount (R$)` e outras taxas. Se usarmos `total (R$)` para prever `rent amount (R$)`, o modelo aprenderia uma simples subtração, resultando em métricas perfeitas mas inúteis na prática. 
- **Data Leakage:** Esse fenômeno é um dos erros mais comuns e graves em machine learning. 
- **Solução:** Nossa variável alvo (`y`) será `rent amount (R$)` e a coluna `total (R$)` será removida do conjunto de features (`X`).

### 1.2. Análise Exploratória de Dados (EDA): Descobrindo Padrões com `pairplot`

- **O "Debugger Visual" para Dados:** A função `pairplot` da biblioteca `seaborn` é uma ferramenta de diagnóstico poderosa. Ela cria uma matriz de visualizações que permite uma compreensão holística das relações e distribuições nos dados.

In [ ]:
# Selecionar um subconjunto de colunas para o pairplot para melhor visualização
colunas_selecionadas = ['area', 'rooms', 'bathroom', 'parking spaces', 'rent amount (R$)']
df_subset = df_aluguel[colunas_selecionadas]

# Gerar o pairplot
sns.pairplot(df_subset, height=2)
plt.show()

#### Interpretando o `pairplot`

- **Diagonal (Histogramas):** Mostram a distribuição de cada variável. Note que `area` e `rent amount (R$)` são fortemente assimétricas à direita (*right-skewed*). A maioria dos imóveis é menor e mais barata, com uma "cauda longa" de imóveis muito grandes e caros. Isso sugere que transformações (como logaritmo) poderiam ajudar modelos lineares.

- **Fora da Diagonal (Gráficos de Dispersão):** Revelam as relações entre pares de variáveis.
    - **`rent_amount_brl` vs. `area`:** Correlação positiva clara. Conforme a área aumenta, o preço do aluguel tende a aumentar.
    - **`rent_amount_brl` vs. `rooms`/`bathroom`:** Também se espera uma correlação positiva, mas com maior dispersão.

- **O `pairplot` como um Motor de Geração de Hipóteses:** A análise visual confirma que uma `Regressão Linear` é um ponto de partida razoável, mas as distribuições assimétricas e a dispersão sugerem que modelos mais robustos, como `Random Forest`, podem ter um desempenho superior.

### 1.3. Preparação dos Dados: Engenharia de Features para Modelos

- **A Necessidade do Pré-processamento:** Modelos de machine learning exigem entradas numéricas e limpas. Esta etapa "traduz" os dados brutos para um formato que os algoritmos possam entender.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Criar uma cópia para evitar alterações no dataframe original
df_processado = df_aluguel.copy()

# Lista de colunas para limpar
cols_to_clean = ['hoa (R$)', 'rent amount (R$)', 'property tax (R$)', 'fire insurance (R$)']

# Remover 'R$' e ',' e converter para float
for col in cols_to_clean:
    df_processado[col] = df_processado[col].replace({'R\$': '', ',': ''}, regex=True).astype(float)

# Tratar a coluna 'floor', substituindo '-' por 0 (assumindo que '-' significa térreo)
df_processado['floor'] = df_processado['floor'].replace('-', '0').astype(int)

# Remover a coluna 'total (R$)' para evitar data leakage
df_processado = df_processado.drop('total (R$)', axis=1)

# Separar features (X) e alvo (y)
X = df_processado.drop('rent amount (R$)', axis=1)
y = df_processado['rent amount (R$)']

# Identificar colunas numéricas e categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Criar o pré-processador com ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Dividir os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho do conjunto de treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de teste: {X_test.shape[0]} amostras")

### 1.4. Modelo 1: A Linha de Base com Regressão Linear

In [ ]:
from sklearn.linear_model import LinearRegression

# Criar o pipeline completo com pré-processamento e modelo
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Treinar o modelo
lr_pipeline.fit(X_train, y_train)

print("Pipeline de Regressão Linear treinado com sucesso!")

### 1.5. Modelo 2: Uma Abordagem Mais Poderosa com Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Criar o pipeline do Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)) # n_jobs=-1 usa todos os cores
])

# Treinar o modelo
rf_pipeline.fit(X_train, y_train)

print("Pipeline de Random Forest Regressor treinado com sucesso!")

### 1.6. Avaliação e Interpretação: Quão Boas São Nossas Predições?

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Fazer predições no conjunto de teste
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# Calcular RMSE e R² para a Regressão Linear
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

# Calcular RMSE e R² para o Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"--- Regressão Linear ---")
print(f"RMSE: R$ {rmse_lr:.2f}")
print(f"R²: {r2_lr:.2f}")
print("\n--- Random Forest Regressor ---")
print(f"RMSE: R$ {rmse_rf:.2f}")
print(f"R²: {r2_rf:.2f}")

---

## Parte 2: Um Estudo Comparativo de Classificadores – Prevendo Reviews de E-commerce

Nesta segunda parte, o foco muda de prever um valor contínuo para classificar um resultado em categorias. Usaremos o famoso dataset da Olist para prever se um review de cliente será positivo ou negativo.

### 2.1. Carregando e Unindo os Dados da Olist

- **Contexto:** O desafio é para uma grande plataforma de e-commerce, a Olist. O objetivo é identificar proativamente os pedidos que têm alta probabilidade de receber uma avaliação negativa.
- **Visão Geral do Dataset:** O dataset da Olist é complexo e distribuído em múltiplas tabelas. Precisaremos combinar informações de vários arquivos CSV para criar um único dataset de análise.

In [ ]:
# NOTA: Você precisa baixar o dataset da Olist do Kaggle e descompactá-lo na mesma pasta.
# Link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

print("Carregando datasets da Olist...")
try:
    orders = pd.read_csv('olist_orders_dataset.csv')
    reviews = pd.read_csv('olist_order_reviews_dataset.csv')
    items = pd.read_csv('olist_order_items_dataset.csv')
    payments = pd.read_csv('olist_order_payments_dataset.csv')
    customers = pd.read_csv('olist_customers_dataset.csv')
    print("Datasets carregados com sucesso.")
except FileNotFoundError:
    print("Erro: Arquivos do dataset Olist não encontrados. Por favor, baixe e descompacte-os do Kaggle.")

# Unindo os dataframes
print("Unindo os dataframes...")
df = orders.merge(reviews, on='order_id')
df = df.merge(items, on='order_id')
df = df.merge(payments, on='order_id')
df = df.merge(customers, on='customer_id')

print("Shape do dataframe unido:", df.shape)
display(df.head())

### 2.2. Engenharia de Features e Pré-processamento

Agora, vamos criar features úteis e preparar os dados para os modelos.

1.  **Criar a Variável Alvo:** Transformaremos o `review_score` (1 a 5) em uma variável binária: `0` para reviews negativos (notas 1, 2, 3) e `1` para positivos (notas 4, 5).
2.  **Criar Features de Tempo:** Calcularemos o tempo de entrega em dias.
3.  **Selecionar Features:** Escolheremos um subconjunto de colunas numéricas para simplificar.
4.  **Limpeza:** Trataremos valores ausentes e converteremos colunas de data para o formato correto.

In [ ]:
# --- Engenharia de Features ---

# 1. Criar a variável alvo
df['review_category'] = df['review_score'].apply(lambda x: 1 if x >= 4 else 0)

# 2. Converter colunas de data para datetime
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. Calcular tempo de entrega
df['delivery_time'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# --- Limpeza e Seleção ---

# 4. Selecionar features para o modelo
features = ['price', 'freight_value', 'payment_value', 'delivery_time']
target = 'review_category'

df_model = df[features + [target]].copy()

# 5. Tratar valores ausentes (preencher com a mediana)
for col in features:
    median_val = df_model[col].median()
    df_model[col].fillna(median_val, inplace=True)

print("Dados prontos para modelagem:")
display(df_model.head())
print("\nDistribuição da variável alvo:")
print(df_model[target].value_counts(normalize=True))

### 2.3. Preparando os Pipelines e Dividindo os Dados

Com os dados limpos, vamos definir os pipelines para cada classificador e dividir os dados em conjuntos de treino e teste.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Separar features (X) e alvo (y)
X_olist = df_model.drop(target, axis=1)
y_olist = df_model[target]

# Dividir em treino e teste
X_train_olist, X_test_olist, y_train_olist, y_test_olist = train_test_split(
    X_olist, y_olist, test_size=0.3, random_state=42, stratify=y_olist # stratify é importante para classes desbalanceadas
)

# Criar pipelines para cada modelo (apenas StandardScaler, pois só temos features numéricas)
dt_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', DecisionTreeClassifier(random_state=42))])
rf_pipeline_cls = Pipeline([('scaler', StandardScaler()), ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))])
svm_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', SVC(random_state=42))])
knn_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', KNeighborsClassifier())])

models = {
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline_cls,
    "SVM": svm_pipeline,
    "KNN": knn_pipeline
}

print("Pipelines de classificação e dados divididos.")

### 2.4. Treinamento e Avaliação com Matriz de Confusão

Agora, vamos treinar cada modelo e avaliá-lo usando o `classification_report` e a `confusion_matrix`.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

for name, model in models.items():
    print(f"--- Treinando e Avaliando: {name} ---")
    
    # Treinar o modelo
    model.fit(X_train_olist, y_train_olist)
    
    # Fazer predições
    y_pred_olist = model.predict(X_test_olist)
    
    # Imprimir o relatório de classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_test_olist, y_pred_olist, target_names=['Negativo (0)', 'Positivo (1)']))
    
    # Gerar e plotar a matriz de confusão
    cm = confusion_matrix(y_test_olist, y_pred_olist)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negativo (0)', 'Positivo (1)'], 
                yticklabels=['Negativo (0)', 'Positivo (1)'])
    plt.xlabel('Predito')
    plt.ylabel('Verdadeiro')
    plt.title(f'Matriz de Confusão - {name}')
    plt.show()
    print("\n" + "="*50 + "\n")

### 2.5. Análise dos Resultados e Escolha do Modelo

- **Interpretação da Matriz de Confusão:** A matriz nos mostra exatamente onde o modelo está acertando e errando. Por exemplo, no canto superior direito (Falso Positivo), vemos quantos reviews negativos foram classificados incorretamente como positivos. No canto inferior esquerdo (Falso Negativo), vemos o erro mais crítico para o negócio: quantos reviews positivos foram classificados como negativos, significando clientes insatisfeitos que não foram identificados.

- **Comparando os Modelos:**
    - **Decision Tree e KNN:** Geralmente são mais rápidos, mas podem ter performance inferior e ser mais sensíveis a ruídos.
    - **Random Forest:** Costuma apresentar um ótimo equilíbrio entre performance e velocidade de treino. Note o bom desempenho tanto para a classe positiva quanto para a negativa.
    - **SVM:** Pode ser muito poderoso, mas seu tempo de treinamento em datasets grandes (como este) pode ser proibitivo para experimentação rápida.

- **Escolha para Produção:** O **Random Forest** se destaca como a escolha mais pragmática. Ele oferece um **recall** robusto para a classe negativa (essencial para identificar clientes insatisfeitos) e mantém uma boa precisão geral, tudo isso com um tempo de treinamento razoável. Ele representa o melhor compromisso entre performance, velocidade e escalabilidade para este problema de negócio.

---

## Parte 3: Desafio Final – Prevendo Preços de Carros Usados

Este exercício final foi projetado para que você aplique todo o conhecimento adquirido em um novo dataset.

### 3.1. O Dataset e Sua Missão

- **Dataset:** "Car Price Prediction Dataset" do Kaggle. Contém features como: `Car_Name`, `Year`, `Selling_Price` (alvo), `Present_Price`, `Kms_Driven`, `Fuel_Type`, `Seller_Type`, `Transmission`, `Owner`.
- **O Desafio:** Você atua como consultor para um marketplace de carros usados. Sua missão é construir e avaliar modelos de regressão para ajudar os vendedores a precificar seus veículos.

### 3.2. Sua Tarefa (O Exercício)

Siga os passos abaixo para completar o desafio:

1.  **Carregar e Explorar:** Carregue o dataset. Realize uma breve EDA. Use `pairplot` ou um `heatmap` de correlação para identificar as features mais correlacionadas com o `Selling_Price`.
2.  **Pré-processar os Dados:** Crie um pipeline de pré-processamento completo usando `ColumnTransformer` para tratar features categóricas e numéricas.
3.  **Treinar e Comparar Modelos:** Escolha pelo menos dois modelos de regressão diferentes (ex: `LinearRegression`, `RandomForestRegressor`, ou experimente `GradientBoostingRegressor`).
4.  **Avaliar a Performance:** Calcule o RMSE e o R² para ambos os modelos no conjunto de teste.
5.  **Escrever uma Conclusão:** Escreva um parágrafo recomendando qual modelo deve ser implantado e por quê, justificando sua escolha com base nas métricas. Forneça a interpretação final para o negócio: "O modelo recomendado erra, em média, X unidades monetárias na sua predição de preço."